# Notebook 02 — Eksploracyjna Analiza Danych (EDA)

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych  
**Autor:** Praca Magisterska  

---

## Cel notebooka

Przeprowadzamy pełną analizę eksploracyjną danych zebranych w Notebooku 01:

1. **Znormalizowane ceny** — BTC i WIG20 na osi 100, z zaznaczonymi kryzysami
2. **Rolling correlation** — okna 30/60/90 dni, dynamika korelacji BTC–WIG20
3. **Scatter returns** — rozkład punktów według epok (pre-COVID, COVID, post-COVID, kryzysy krypto)
4. **Rozkłady zwrotów** — histogramy + QQ-plot (grube ogony vs. normalność)
5. **Heatmapa korelacji** — Pearson i Spearman
6. **Analiza wolumenów** — BTC i WIG20 z rolling avg
7. **Porównanie zmienności** — rolling realized vol 21d

## Wyniki
Wszystkie wykresy zapisywane do `results/figures/` w PNG (300 dpi) i PDF.

In [1]:
# ============================================================
# Reprodukowalność + Importy
# ============================================================
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde
import warnings
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
warnings.filterwarnings('ignore')

# ── Ścieżki ─────────────────────────────────────────────────
BASE_DIR = Path('.')
DATA_PROC = BASE_DIR / 'data' / 'processed'
RESULTS_TABLES = BASE_DIR / 'results' / 'tables'
RESULTS_FIGS = BASE_DIR / 'results' / 'figures'

for d in [RESULTS_TABLES, RESULTS_FIGS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Styl wykresów ────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.size'] = 11
mpl.rcParams['axes.titlesize'] = 13
mpl.rcParams['axes.labelsize'] = 11
mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['savefig.dpi'] = 300

COLOR_BTC = '#F7931A'
COLOR_WIG = '#003087'
COLOR_HRI = '#C0392B'
COLOR_VIX = '#27AE60'
COLOR_NEUTRAL = '#7F8C8D'

# ── Zdarzenia kryzysowe ──────────────────────────────────────
CRISIS_EVENTS = {
    '2020-03-12': ('COVID-19 Crash', '#E74C3C'),
    '2022-05-09': ('Terra/LUNA', '#E67E22'),
    '2022-11-08': ('FTX Bankruptcy', '#8E44AD'),
    '2023-03-10': ('SVB Bank Run', '#2980B9'),
}

print("Biblioteki zaladowane.")
print(f"Matplotlib: v{mpl.__version__}")
print(f"Seaborn: v{sns.__version__}")

Biblioteki zaladowane.
Matplotlib: v3.10.0
Seaborn: v0.13.2


In [2]:
# ============================================================
# Wczytanie danych z Notebooka 01
# ============================================================

master_path = DATA_PROC / 'master_dataset.csv'
if not master_path.exists():
    raise FileNotFoundError(
        f"Brak pliku {master_path}. "
        "Uruchom najpierw Notebook 01 (01_setup_and_data_download.ipynb)"
    )

df = pd.read_csv(master_path, index_col='Date', parse_dates=True)

# Oblicz log-returns jeśli nie ma
if 'btc_log_return' not in df.columns:
    df['btc_log_return'] = np.log(df['btc_close'] / df['btc_close'].shift(1))
if 'wig_log_return' not in df.columns:
    df['wig_log_return'] = np.log(df['wig_close'] / df['wig_close'].shift(1))

# Usuń outlier-flagi z numeric obliczeń
flag_cols = [c for c in df.columns if c.startswith('flag_')]

print(f"Master dataset wczytany:")
print(f"  Kształt: {df.shape}")
print(f"  Zakres: {df.index.min().date()} → {df.index.max().date()}")
print(f"  Kolumny: {list(df.columns)}")

Master dataset wczytany:
  Kształt: (1906, 24)
  Zakres: 2019-01-02 → 2026-08-14
  Kolumny: ['wig_open', 'wig_high', 'wig_low', 'wig_close', 'wig_volume', 'btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'vix', 'dxy', 'fed_rate', 'flag_btc_outlier', 'flag_wig_outlier', 'btc_log_return', 'wig_log_return', 'vix_change', 'dxy_change', 'btc_1h_return_mean', 'btc_1h_return_std', 'btc_1h_return_min', 'btc_1h_return_max', 'btc_1h_n_obs']


## Figura 1: Znormalizowane ceny BTC i WIG20

Normalizujemy oba indeksy do bazy 100 (01-01-2019) i umieszczamy na wspólnym wykresie z dwiema osiami Y. Zaznaczamy główne zdarzenia kryzysowe pionowymi liniami.

**Uzasadnienie:** Normalizacja do bazy 100 pozwala na bezpośrednie porównanie dynamiki procentowej obu aktywów mimo różnych skal cenowych (BTC w USD, WIG20 w punktach).

In [3]:
# ============================================================
# FIGURA 1: Znormalizowane ceny BTC i WIG20 (baza=100)
# ============================================================

fig, ax1 = plt.subplots(figsize=(14, 6))

# Normalizacja do bazy 100
btc_norm = df['btc_close'] / df['btc_close'].iloc[0] * 100
wig_norm = df['wig_close'] / df['wig_close'].iloc[0] * 100

# Oś lewa — BTC
ax1.plot(df.index, btc_norm, color=COLOR_BTC, linewidth=1.2,
         label='BTC/USD (lewa oś)', alpha=0.9)
ax1.set_xlabel('Data')
ax1.set_ylabel('BTC/USD (znormalizowany, 1 sty 2019 = 100)', color=COLOR_BTC)
ax1.tick_params(axis='y', labelcolor=COLOR_BTC)
ax1.set_yscale('log')  # log-skala dla BTC ze względu na ogromny wzrost

# Oś prawa — WIG20
ax2 = ax1.twinx()
ax2.plot(df.index, wig_norm, color=COLOR_WIG, linewidth=1.2,
         label='WIG20 (prawa oś)', alpha=0.9)
ax2.set_ylabel('WIG20 (znormalizowany, 1 sty 2019 = 100)', color=COLOR_WIG)
ax2.tick_params(axis='y', labelcolor=COLOR_WIG)

# Zaznaczenie zdarzeń kryzysowych
for date_str, (label, color) in CRISIS_EVENTS.items():
    event_date = pd.Timestamp(date_str)
    if df.index.min() <= event_date <= df.index.max():
        ax1.axvline(event_date, color=color, alpha=0.7, linewidth=1.5, linestyle='--')
        y_pos = ax1.get_ylim()[0] * 1.1  # pozycja tekstu blisko dolnej osi
        ax1.text(event_date + pd.Timedelta(days=5), 
                 btc_norm.min() * 1.05,
                 label, rotation=90, fontsize=8, color=color,
                 va='bottom', ha='left')

# Obszary cieniowane — okresy kryzysowe
crisis_periods = [
    ('2020-02-20', '2020-04-01', 'COVID-19', 0.05),
    ('2022-04-01', '2022-06-30', 'Krypto winter 2022', 0.04),
    ('2022-10-15', '2022-12-01', 'FTX & aftermath', 0.04),
]
for start, end, label, alpha in crisis_periods:
    ax1.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                alpha=alpha, color='red', label=f'_nolegend_{label}')

# Linia pozioma na 100 (punkt startu)
ax2.axhline(100, color='gray', linestyle=':', linewidth=0.8, alpha=0.5)

# Legenda
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

ax1.set_title('Znormalizowane ceny BTC/USD i WIG20 (baza = 100, 01.01.2019)\n'
              'Linie przerywane: kluczowe zdarzenia kryzysowe', fontsize=13, pad=12)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
for ext in ['png', 'pdf']:
    fpath = RESULTS_FIGS / f'fig_01_price_series.{ext}'
    plt.savefig(fpath, dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_01_price_series.png + .pdf")

Zapisano: fig_01_price_series.png + .pdf


## Figura 2: Rolling Correlation BTC vs WIG20

Obliczamy kroczącą korelację log-zwrotów w oknach 30, 60 i 90 dni. Korelacja w czasie nie jest stała — wzrasta szczególnie w okresach stresu rynkowego, co jest dowodem efektu zarażenia (contagion).

**Hipoteza badawcza:** W momentach kryzysowych korelacja BTC–WIG20 wzrasta (wspólna wyprzedaż aktywów ryzykownych).

In [4]:
# ============================================================
# FIGURA 2: Rolling Correlation (30d, 60d, 90d)
# ============================================================

# Oblicz rolling correlations
btc_ret = df['btc_log_return'].dropna()
wig_ret = df['wig_log_return'].dropna()

# Wyrównaj indeksy (oba szeregi muszą mieć te same daty)
common_idx = btc_ret.index.intersection(wig_ret.index)
btc_ret_aligned = btc_ret.loc[common_idx]
wig_ret_aligned = wig_ret.loc[common_idx]

# Rolling correlations
windows = [30, 60, 90]
colors_win = ['#E74C3C', '#F39C12', '#27AE60']
alphas_win = [0.9, 0.8, 0.7]
lw_win = [1.5, 1.3, 1.1]

roll_corrs = {}
for w in windows:
    roll_corrs[w] = btc_ret_aligned.rolling(window=w).corr(wig_ret_aligned)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})

# ── Górny panel: rolling correlations ─────────────────────────
for w, color, alpha, lw in zip(windows, colors_win, alphas_win, lw_win):
    ax1.plot(roll_corrs[w].index, roll_corrs[w],
             label=f'Korelacja {w}d', color=color, alpha=alpha, linewidth=lw)

ax1.axhline(0, color='black', linewidth=0.8, linestyle='-')
ax1.axhline(0.3, color='gray', linewidth=0.5, linestyle=':', alpha=0.7)
ax1.axhline(-0.3, color='gray', linewidth=0.5, linestyle=':', alpha=0.7)

# Zaznacz zdarzenia kryzysowe
for date_str, (label, color) in CRISIS_EVENTS.items():
    event_date = pd.Timestamp(date_str)
    if df.index.min() <= event_date <= df.index.max():
        ax1.axvline(event_date, color=color, alpha=0.6, linewidth=1.2, linestyle='--')
        ax1.text(event_date + pd.Timedelta(days=3), 0.85, label,
                 rotation=90, fontsize=7.5, color=color, va='top')

# Cieniuj obszary gdzie korelacja > 0.3 (silna pozytywna)
corr_90 = roll_corrs[90].reindex(df.index)
ax1.fill_between(corr_90.index, corr_90, 0,
                 where=(corr_90 > 0.3), alpha=0.15, color='red',
                 label='Korelacja > 0.3 (90d)')

ax1.set_ylabel('Korelacja Pearsona (rolling)')
ax1.set_ylim(-0.8, 1.0)
ax1.set_title('Krocząca korelacja log-zwrotów BTC/USD vs WIG20\n'
              'Okna: 30, 60, 90 dni', fontsize=13)
ax1.legend(loc='lower right', fontsize=9)

# ── Dolny panel: WIG20 zwroty (kontekst) ──────────────────────
ax2.bar(wig_ret_aligned.index, wig_ret_aligned * 100,
        color=COLOR_WIG, alpha=0.4, width=1)
ax2.axhline(0, color='black', linewidth=0.5)
ax2.set_ylabel('WIG20\nzwroty (%)')
ax2.set_xlabel('Data')

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_02_rolling_correlation.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_02_rolling_correlation.png + .pdf")

# Statystyki rolling correlation
print("\nStatystyki rolling correlation (30d):")
rc30 = roll_corrs[30].dropna()
print(f"  Średnia:   {rc30.mean():.4f}")
print(f"  Mediana:   {rc30.median():.4f}")
print(f"  Std:       {rc30.std():.4f}")
print(f"  Min:       {rc30.min():.4f} (data: {rc30.idxmin().date()})")
print(f"  Max:       {rc30.max():.4f} (data: {rc30.idxmax().date()})")
print(f"  % dodatnich: {(rc30 > 0).mean()*100:.1f}%")

Zapisano: fig_02_rolling_correlation.png + .pdf

Statystyki rolling correlation (30d):
  Średnia:   0.1396
  Mediana:   0.1534
  Std:       0.2311
  Min:       -0.4759 (data: 2020-02-04)
  Max:       0.7654 (data: 2020-03-19)
  % dodatnich: 73.0%


## Figura 3: Scatter plot zwrotów według epoki

Dzielimy obserwacje na 4 okresy i kolorujemy punkty:
- **Pre-COVID** (2019-01 do 2020-02): względna normalność
- **COVID** (2020-03 do 2020-08): kryzys zdrowotny, silna korelacja
- **Crypto Winter** (2022-05 do 2022-12): kryzysy Luna, FTX
- **Post-FTX** (2023-01 do 2024-12): odbudowa rynku

In [5]:
# ============================================================
# FIGURA 3: Scatter plot zwrotów BTC vs WIG20 według epoki
# ============================================================

# Definicja epok
def assign_epoch(date):
    if date < pd.Timestamp('2020-03-01'):
        return 'Pre-COVID (2019-2020.02)'
    elif date <= pd.Timestamp('2020-08-31'):
        return 'COVID Crash (2020.03-08)'
    elif date < pd.Timestamp('2022-05-01'):
        return 'Bull Market (2020.09-2022.04)'
    elif date <= pd.Timestamp('2022-12-31'):
        return 'Crypto Winter (2022.05-12)'
    else:
        return 'Odbudowa i nowy cykl (2023-2026)'

df_ret = df[['btc_log_return', 'wig_log_return']].dropna().copy()
df_ret['epoch'] = df_ret.index.map(assign_epoch)

epoch_colors = {
    'Pre-COVID (2019-2020.02)': '#3498DB',
    'COVID Crash (2020.03-08)': '#E74C3C',
    'Bull Market (2020.09-2022.04)': '#27AE60',
    'Crypto Winter (2022.05-12)': '#E67E22',
    'Odbudowa i nowy cykl (2023-2026)': '#8E44AD'
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Lewy panel: scatter z kolorowaniem epoki ──────────────────
ax = axes[0]
for epoch, color in epoch_colors.items():
    mask = df_ret['epoch'] == epoch
    subset = df_ret[mask]
    n = len(subset)
    ax.scatter(subset['btc_log_return'] * 100,
               subset['wig_log_return'] * 100,
               c=color, alpha=0.4, s=12,
               label=f'{epoch} (n={n})')

# Dodaj linię trendu (OLS)
x_all = df_ret['btc_log_return'].values * 100
y_all = df_ret['wig_log_return'].values * 100
mask_valid = ~np.isnan(x_all) & ~np.isnan(y_all)
z = np.polyfit(x_all[mask_valid], y_all[mask_valid], 1)
p = np.poly1d(z)
x_line = np.linspace(x_all[mask_valid].min(), x_all[mask_valid].max(), 100)
ax.plot(x_line, p(x_line), 'k--', linewidth=1.2, alpha=0.7,
        label=f'OLS: y={z[0]:.3f}x+{z[1]:.3f}')

ax.axhline(0, color='gray', linewidth=0.5, alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.5, alpha=0.5)
ax.set_xlabel('BTC/USD log-zwrot (%)')
ax.set_ylabel('WIG20 log-zwrot (%)')
ax.set_title('Zwroty BTC vs WIG20 — kolorowanie według epoki')
ax.legend(fontsize=7, loc='upper left')

# Oblicz i wyświetl korelację
corr_pearson, p_pearson = stats.pearsonr(x_all[mask_valid], y_all[mask_valid])
corr_spearman, p_spearman = stats.spearmanr(x_all[mask_valid], y_all[mask_valid])
ax.text(0.97, 0.05,
        f'r_Pearson = {corr_pearson:.3f} (p={p_pearson:.3e})\n'
        f'r_Spearman = {corr_spearman:.3f} (p={p_spearman:.3e})',
        transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# ── Prawy panel: korelacja per epoka ──────────────────────────
ax2 = axes[1]
epoch_corrs = []
for epoch in epoch_colors.keys():
    mask = df_ret['epoch'] == epoch
    subset = df_ret[mask]
    if len(subset) > 10:
        r, p = stats.pearsonr(subset['btc_log_return'], subset['wig_log_return'])
        rs, ps = stats.spearmanr(subset['btc_log_return'], subset['wig_log_return'])
        epoch_corrs.append({
            'epoch': epoch, 'pearson': r, 'spearman': rs,
            'p_pearson': p, 'n': len(subset)
        })

ec_df = pd.DataFrame(epoch_corrs)
x_pos = range(len(ec_df))
bar_width = 0.35

bars1 = ax2.bar([x - bar_width/2 for x in x_pos], ec_df['pearson'],
                width=bar_width, label='Pearson r',
                color=[epoch_colors[e] for e in ec_df['epoch']], alpha=0.8)
bars2 = ax2.bar([x + bar_width/2 for x in x_pos], ec_df['spearman'],
                width=bar_width, label='Spearman ρ',
                color=[epoch_colors[e] for e in ec_df['epoch']], alpha=0.4,
                hatch='//')

ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xticks(list(x_pos))
ax2.set_xticklabels([e.split('(')[0].strip() for e in ec_df['epoch']],
                     rotation=30, ha='right', fontsize=8)
ax2.set_ylabel('Korelacja')
ax2.set_title('Korelacja BTC–WIG20 per epoka')
ax2.legend(fontsize=9)
ax2.set_ylim(-0.5, 0.8)

# Dodaj wartości nad słupkami
for bar, val in zip(bars1, ec_df['pearson']):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01 * np.sign(val),
             f'{val:.2f}', ha='center', va='bottom' if val > 0 else 'top', fontsize=8)

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_03_scatter_returns.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_03_scatter_returns.png + .pdf")

# Zapis korelacji per epoka
ec_df.to_csv(RESULTS_TABLES / '02_correlation_by_epoch.csv', index=False)
print(ec_df.to_string(index=False))

Zapisano: fig_03_scatter_returns.png + .pdf
                           epoch   pearson  spearman    p_pearson   n
        Pre-COVID (2019-2020.02) -0.054139 -0.061463 3.599552e-01 288
        COVID Crash (2020.03-08)  0.595342  0.192380 1.560206e-13 127
   Bull Market (2020.09-2022.04)  0.195970  0.178733 5.602747e-05 417
      Crypto Winter (2022.05-12)  0.356773  0.324929 1.923829e-06 169
Odbudowa i nowy cykl (2023-2026)  0.092525  0.124704 5.368721e-03 904


## Figura 4: Rozkłady zwrotów — histogramy i QQ-ploty

Testujemy empirycznie założenie normalności rozkładu zwrotów. Oczekujemy:
- **Leptokurtoza** (excess kurtosis > 0) — grube ogony, więcej ekstremalnych zdarzeń niż przewiduje rozkład normalny
- **Ujemna skośność** — asymetria w kierunku strat (dotyczy szczególnie WIG20)

W QQ-plotach odchylenia od linii 45° widoczne na krańcach potwierdzają gruboogonowość.

In [6]:
# ============================================================
# FIGURA 4: Rozkłady zwrotów — 2x2 subplot
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Rozkłady dziennych log-zwrotów BTC/USD i WIG20', fontsize=14, y=1.02)

btc_r = df['btc_log_return'].dropna() * 100  # procenty
wig_r = df['wig_log_return'].dropna() * 100

assets = [
    (btc_r, 'BTC/USD', COLOR_BTC),
    (wig_r, 'WIG20', COLOR_WIG)
]

for col, (returns, name, color) in enumerate(assets):
    
    # ── Górny rząd: Histogramy ─────────────────────────────────
    ax = axes[0, col]
    
    # Histogram
    n, bins, patches = ax.hist(returns, bins=80, density=True,
                                color=color, alpha=0.65, edgecolor='none',
                                label=f'{name} (n={len(returns)})')
    
    # KDE empiryczne
    kde = gaussian_kde(returns)
    x_range = np.linspace(returns.min(), returns.max(), 300)
    ax.plot(x_range, kde(x_range), color=color, linewidth=2, alpha=0.9, label='KDE empiryczne')
    
    # Rozkład normalny dopasowany
    mu, sigma = returns.mean(), returns.std()
    normal_curve = stats.norm.pdf(x_range, mu, sigma)
    ax.plot(x_range, normal_curve, 'k--', linewidth=1.5, alpha=0.7, label='Rozkład normalny')
    
    # Statystyki
    kurt = stats.kurtosis(returns)  # excess kurtosis
    skew = stats.skew(returns)
    jb, jb_p = stats.jarque_bera(returns)
    
    stats_text = (f'μ = {mu:.3f}%\n'
                  f'σ = {sigma:.3f}%\n'
                  f'Skośność = {skew:.3f}\n'
                  f'Kurtoza nadmiarowa = {kurt:.2f}\n'
                  f'JB p-value = {jb_p:.2e}')
    ax.text(0.97, 0.97, stats_text, transform=ax.transAxes,
            fontsize=8, va='top', ha='right',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    
    ax.set_xlabel('Log-zwrot (%)')
    ax.set_ylabel('Gęstość')
    ax.set_title(f'{name} — Histogram log-zwrotów')
    ax.legend(fontsize=8)
    
    # Ogranicz oś x do rozsądnego zakresu (wyklucz ekstremalne outliers z wykresu)
    q01 = np.percentile(returns, 0.5)
    q99 = np.percentile(returns, 99.5)
    ax.set_xlim(q01 * 1.2, q99 * 1.2)
    
    # ── Dolny rząd: QQ-ploty ───────────────────────────────────
    ax_qq = axes[1, col]
    
    # Quantile-Quantile plot vs rozkład normalny
    (quantiles, values), (slope, intercept, r) = stats.probplot(returns, dist='norm')
    
    ax_qq.scatter(quantiles, values, color=color, alpha=0.4, s=8, label=name)
    
    # Linia referencyjna (normalność)
    q_line = np.linspace(quantiles.min(), quantiles.max(), 100)
    ax_qq.plot(q_line, slope * q_line + intercept, 'k-', linewidth=1.5,
               label='Rozkład normalny', alpha=0.8)
    
    ax_qq.set_xlabel('Kwantyle teoretyczne (N(0,1))')
    ax_qq.set_ylabel('Kwantyle empiryczne')
    ax_qq.set_title(f'{name} — QQ-plot (vs. normalny)')
    ax_qq.legend(fontsize=8)
    
    ax_qq.text(0.03, 0.97,
               f'Grube ogony widoczne\nna krańcach wykresu',
               transform=ax_qq.transAxes, fontsize=8, va='top',
               color='red', alpha=0.7)

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_04_return_distributions.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_04_return_distributions.png + .pdf")

Zapisano: fig_04_return_distributions.png + .pdf


## Figura 5: Heatmapa korelacji Pearsona i Spearmana

In [7]:
# ============================================================
# FIGURA 5: Heatmapa korelacji Pearsona i Spearmana
# ============================================================

# Wybierz kolumny do analizy korelacji
corr_cols = ['btc_log_return', 'wig_log_return']
col_labels = ['BTC log-return', 'WIG20 log-return']

if 'vix' in df.columns:
    df['vix_change'] = df['vix'].pct_change()
    corr_cols.append('vix_change')
    col_labels.append('VIX zmiana (%)')
    
if 'dxy' in df.columns:
    df['dxy_change'] = df['dxy'].pct_change()
    corr_cols.append('dxy_change')
    col_labels.append('DXY zmiana (%)')

if 'btc_volume' in df.columns:
    df['btc_vol_change'] = df['btc_volume'].pct_change()
    corr_cols.append('btc_vol_change')
    col_labels.append('BTC wolumen (zmiana)')

# Dataset do korelacji
df_corr = df[corr_cols].dropna().copy()
df_corr.columns = col_labels

# Oblicz macierze korelacji
pearson_corr = df_corr.corr(method='pearson')
spearman_corr = df_corr.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

heatmap_params = {
    'annot': True,
    'fmt': '.3f',
    'cmap': 'RdBu_r',
    'vmin': -1, 'vmax': 1,
    'linewidths': 0.5,
    'annot_kws': {'size': 10},
    'square': True
}

# Maska dla górnego trójkąta
mask = np.triu(np.ones_like(pearson_corr), k=1).astype(bool)

sns.heatmap(pearson_corr, ax=axes[0], mask=mask, **heatmap_params)
axes[0].set_title('Korelacja Pearsona\n(liniowa)', fontsize=12)

sns.heatmap(spearman_corr, ax=axes[1], mask=mask, **heatmap_params)
axes[1].set_title('Korelacja Spearmana\n(rangowa, odporna na outliers)', fontsize=12)

for ax in axes:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=9)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)

fig.suptitle('Macierze korelacji — log-zwroty i zmienne makro (2019–2024)',
             fontsize=13, y=1.02)
plt.tight_layout()

for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_05_correlation_heatmap.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_05_correlation_heatmap.png + .pdf")

# Zapis macierzy korelacji
pearson_corr.to_csv(RESULTS_TABLES / '02_pearson_correlation.csv', float_format='%.4f')
spearman_corr.to_csv(RESULTS_TABLES / '02_spearman_correlation.csv', float_format='%.4f')
print("Zapisano tabele korelacji.")

Zapisano: fig_05_correlation_heatmap.png + .pdf
Zapisano tabele korelacji.


## Figura 6: Analiza wolumenów

In [8]:
# ============================================================
# FIGURA 6: Wolumeny BTC i WIG20 z rolling average 30d
# ============================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Wolumeny obrotu — BTC i WIG20 (2019–2024)', fontsize=13, y=1.01)

# ── BTC wolumen ──────────────────────────────────────────────
ax = axes[0]
if 'btc_volume' in df.columns:
    btc_vol = df['btc_volume'].copy()
    btc_vol_roll = btc_vol.rolling(30).mean()
    
    ax.bar(btc_vol.index, btc_vol / 1e9, color=COLOR_BTC, alpha=0.4, width=1,
           label='Dzienny wolumen BTC')
    ax.plot(btc_vol_roll.index, btc_vol_roll / 1e9, color=COLOR_BTC,
            linewidth=1.5, label='Rolling avg 30d')
    ax.set_ylabel('Wolumen (mld USD)')
    ax.set_title('BTC/USD — Dzienny wolumen obrotu')
    ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'Dane wolumenu BTC niedostępne',
            ha='center', va='center', transform=ax.transAxes, fontsize=12)

# Oznacz zdarzenia
for date_str, (label, color) in CRISIS_EVENTS.items():
    event_date = pd.Timestamp(date_str)
    if df.index.min() <= event_date <= df.index.max():
        axes[0].axvline(event_date, color=color, alpha=0.6, linewidth=1.2, linestyle='--')

# ── WIG20 wolumen ─────────────────────────────────────────────
ax2 = axes[1]
if 'wig_volume' in df.columns:
    wig_vol = df['wig_volume'].copy()
    wig_vol_valid = wig_vol[wig_vol > 0]
    wig_vol_roll = wig_vol.rolling(30).mean()
    
    ax2.bar(wig_vol.index, wig_vol / 1e6, color=COLOR_WIG, alpha=0.4, width=1,
            label='Dzienny wolumen WIG20')
    ax2.plot(wig_vol_roll.index, wig_vol_roll / 1e6, color=COLOR_WIG,
             linewidth=1.5, label='Rolling avg 30d')
    ax2.set_ylabel('Wolumen (mln)')
    ax2.set_title('WIG20 — Dzienny wolumen')
    ax2.legend(fontsize=9)
else:
    ax2.text(0.5, 0.5, 'Dane wolumenu WIG20 niedostępne\n(indeksy giełdowe często nie mają wolumenu)',
             ha='center', va='center', transform=ax2.transAxes, fontsize=10,
             color='gray')
    ax2.set_title('WIG20 — Wolumen (niedostępny dla indeksów)')

for date_str, (label, color) in CRISIS_EVENTS.items():
    event_date = pd.Timestamp(date_str)
    if df.index.min() <= event_date <= df.index.max():
        axes[1].axvline(event_date, color=color, alpha=0.6, linewidth=1.2, linestyle='--')

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_06_volume_analysis.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_06_volume_analysis.png + .pdf")

Zapisano: fig_06_volume_analysis.png + .pdf


## Figura 7: Porównanie zmienności BTC vs WIG20

Obliczamy **zrealizowaną zmienność** (rolling std log-zwrotów × √252 × 100%) w oknie 21 dni. Zmienność BTC jest wielokrotnie wyższa niż WIG20, ale w momentach kryzysowych wzrastają synchronicznie — kolejny dowód transmisji ryzyka.

In [9]:
# ============================================================
# FIGURA 7: Rolling Realized Volatility (21d)
# ============================================================

# Rolling realized volatility (annualized)
btc_vol_21 = df['btc_log_return'].rolling(21).std() * np.sqrt(252) * 100
wig_vol_21 = df['wig_log_return'].rolling(21).std() * np.sqrt(252) * 100

if 'vix' in df.columns:
    vix_series = df['vix']

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle('Porównanie zrealizowanej zmienności BTC i WIG20\n'
             'Rolling 21 dni, annualizowana', fontsize=13)

# ── Górny panel: zmienność ────────────────────────────────────
ax = axes[0]
ax.plot(btc_vol_21.index, btc_vol_21, color=COLOR_BTC, linewidth=1.2,
        label='BTC vol 21d', alpha=0.9)
ax.plot(wig_vol_21.index, wig_vol_21, color=COLOR_WIG, linewidth=1.2,
        label='WIG20 vol 21d', alpha=0.9)

# Wypełnij pod BTC vol (żeby widać było skalę)
ax.fill_between(btc_vol_21.index, btc_vol_21, wig_vol_21,
                where=(btc_vol_21 > wig_vol_21),
                alpha=0.08, color=COLOR_BTC, label='BTC > WIG20')

# Zaznacz zdarzenia kryzysowe i VIX spikes
for date_str, (label, color) in CRISIS_EVENTS.items():
    event_date = pd.Timestamp(date_str)
    if df.index.min() <= event_date <= df.index.max():
        ax.axvline(event_date, color=color, alpha=0.6, linewidth=1.5, linestyle='--')
        ax.text(event_date + pd.Timedelta(days=5),
                ax.get_ylim()[1] * 0.9 if ax.get_ylim()[1] > 0 else 100,
                label, rotation=90, fontsize=7.5, color=color, va='top')

# Linie poziome — poziomy zmienności
for level, label_v, ls in [(50, '50% vol', ':'), (100, '100% vol', '--'), (150, '150% vol', '-')]:
    ax.axhline(level, color='gray', linewidth=0.5, linestyle=ls, alpha=0.5)
    ax.text(df.index.min(), level + 2, label_v, fontsize=7, color='gray')

ax.set_ylabel('Zrealizowana zmienność annualizowana (%)')
ax.legend(loc='upper right', fontsize=9)

# ── Dolny panel: ratio BTC vol / WIG vol ──────────────────────
ax2 = axes[1]
vol_ratio = btc_vol_21 / wig_vol_21
ax2.plot(vol_ratio.index, vol_ratio, color='purple', linewidth=0.8, alpha=0.8)
ax2.axhline(1, color='black', linewidth=0.8)
ax2.fill_between(vol_ratio.index, vol_ratio, 1,
                 where=(vol_ratio > 1), alpha=0.2, color='orange',
                 label='BTC bardziej zmienny')
ax2.set_ylabel('Ratio\nBTC/WIG vol')
ax2.set_xlabel('Data')
ax2.legend(fontsize=8)
ax2.set_ylim(0, vol_ratio.quantile(0.99) * 1.1)  # przytnij ekstremalne wartości

ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
for ext in ['png', 'pdf']:
    plt.savefig(RESULTS_FIGS / f'fig_07_volatility_comparison.{ext}',
                dpi=300 if ext=='png' else 150, bbox_inches='tight')
plt.show()
print("Zapisano: fig_07_volatility_comparison.png + .pdf")

# Podsumowanie zmienności
print("\nŚrednia zmienność annualizowana (2019-2024):")
print(f"  BTC:   {btc_vol_21.mean():.1f}%")
print(f"  WIG20: {wig_vol_21.mean():.1f}%")
print(f"  Ratio BTC/WIG20: {(btc_vol_21/wig_vol_21).mean():.1f}x")

Zapisano: fig_07_volatility_comparison.png + .pdf

Średnia zmienność annualizowana (2019-2024):
  BTC:   57.6%
  WIG20: 21.5%
  Ratio BTC/WIG20: 2.9x


In [10]:
# ============================================================
# DODATKOWA ANALIZA: Rozkład korelacji w czasie
# ============================================================

# Tabela statystyk rolling correlation per rok
roll_corr_30 = df['btc_log_return'].rolling(30).corr(df['wig_log_return'])

yearly_corr = pd.DataFrame()
for year in range(2019, 2025):
    year_data = roll_corr_30[roll_corr_30.index.year == year].dropna()
    if len(year_data) > 0:
        yearly_corr.loc[year, 'Srednia'] = year_data.mean()
        yearly_corr.loc[year, 'Mediana'] = year_data.median()
        yearly_corr.loc[year, 'Std'] = year_data.std()
        yearly_corr.loc[year, 'Min'] = year_data.min()
        yearly_corr.loc[year, 'Max'] = year_data.max()
        yearly_corr.loc[year, 'Pct_dodatnich (%)'] = (year_data > 0).mean() * 100

print("Rolling correlation 30d BTC–WIG20 per rok:")
print(yearly_corr.round(4).to_string())

yearly_corr.to_csv(RESULTS_TABLES / '02_rolling_correlation_yearly.csv', float_format='%.4f')
print(f"\nZapisano: {RESULTS_TABLES / '02_rolling_correlation_yearly.csv'}")

Rolling correlation 30d BTC–WIG20 per rok:
      Srednia  Mediana     Std     Min     Max  Pct_dodatnich (%)
2019  -0.0428  -0.0459  0.2108 -0.4370  0.4202            38.9908
2020   0.1566   0.1632  0.2934 -0.4759  0.7654            77.3810
2021   0.2550   0.2603  0.1792 -0.1725  0.5991            94.4223
2022   0.2786   0.2781  0.1591 -0.1285  0.5651            96.4143
2023   0.0735   0.0788  0.1540 -0.2598  0.4038            68.4000
2024   0.1220   0.1056  0.2794 -0.4202  0.6515            58.2329

Zapisano: results/tables/02_rolling_correlation_yearly.csv


In [11]:
# ============================================================
# PODSUMOWANIE NOTEBOOKA 02
# ============================================================

print("=" * 60)
print(f"Notebook 02 zakończony: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

saved_files = [
    RESULTS_FIGS / 'fig_01_price_series.png',
    RESULTS_FIGS / 'fig_01_price_series.pdf',
    RESULTS_FIGS / 'fig_02_rolling_correlation.png',
    RESULTS_FIGS / 'fig_02_rolling_correlation.pdf',
    RESULTS_FIGS / 'fig_03_scatter_returns.png',
    RESULTS_FIGS / 'fig_04_return_distributions.png',
    RESULTS_FIGS / 'fig_05_correlation_heatmap.png',
    RESULTS_FIGS / 'fig_06_volume_analysis.png',
    RESULTS_FIGS / 'fig_07_volatility_comparison.png',
    RESULTS_TABLES / '02_correlation_by_epoch.csv',
    RESULTS_TABLES / '02_pearson_correlation.csv',
    RESULTS_TABLES / '02_spearman_correlation.csv',
    RESULTS_TABLES / '02_rolling_correlation_yearly.csv',
]

print("\nZapisane pliki:")
for f in saved_files:
    exists = f.exists()
    status = 'OK' if exists else 'BRAK'
    size = f"{f.stat().st_size / 1024:.1f} KB" if exists else '---'
    print(f"  [{status}] {f.name:<50} ({size})")

print("\n" + "=" * 60)
print("GOTOWE! Przejdź do Notebook 03 — Testy Statystyczne.")
print("=" * 60)

Notebook 02 zakończony: 2026-08-18 20:33:37

Zapisane pliki:
  [OK] fig_01_price_series.png                            (520.4 KB)
  [OK] fig_01_price_series.pdf                            (57.8 KB)
  [OK] fig_02_rolling_correlation.png                     (853.3 KB)
  [OK] fig_02_rolling_correlation.pdf                     (122.9 KB)
  [OK] fig_03_scatter_returns.png                         (609.2 KB)
  [OK] fig_04_return_distributions.png                    (612.6 KB)
  [OK] fig_05_correlation_heatmap.png                     (292.2 KB)
  [OK] fig_06_volume_analysis.png                         (412.8 KB)
  [OK] fig_07_volatility_comparison.png                   (877.4 KB)
  [OK] 02_correlation_by_epoch.csv                        (0.5 KB)
  [OK] 02_pearson_correlation.csv                         (0.3 KB)
  [OK] 02_spearman_correlation.csv                        (0.3 KB)
  [OK] 02_rolling_correlation_yearly.csv                  (0.3 KB)

GOTOWE! Przejdź do Notebook 03 — Testy Statystyczn